# Protein

In [ ]:
import os
import sys
from pathlib import Path

for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[name] = "1"

ROOT = Path.cwd().resolve()
while not (ROOT / "utils.py").is_file():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Open this notebook inside the code folder.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from realdata.run_realdata import run_realdata
from utils import ratio_table
from IPython.display import HTML


## Preprocessing

In [ ]:
import pandas as pd

DATA_DIR = ROOT / "realdata" / "data" / "protein"  # Folder for this dataset's source and processed files.

if not (DATA_DIR / "CASP_processed.csv").is_file():
    raw = pd.read_csv(DATA_DIR / "CASP.csv")

    FEATURES = [f"F{i}" for i in range(1, 10)]  # Input columns used to predict the response.

    data = raw[FEATURES + ["RMSD"]].dropna()  # RMSD is the response to predict.
    assert len(data) == 45730
    data.to_csv(DATA_DIR / "CASP_processed.csv", index=False)

In [ ]:
labeled_sizes = range(100, 1001, 100)  # Labeled sample sizes: 100, 200, ..., 1000.
N = 5000  # Number of unlabeled observations (X only).
reps = 100  # Number of repeated experiments for each setting.
seed = 202607196  # Random seed for the first repetition.
jobs = 12  # Maximum number of repetitions run at the same time.
estimators = [  # Methods to fit to each repeated sample.
    "ata_sbf",
    "ata_csbf",
    "lp_nw_mode_2", "lp_ll_mode_2",  # Real-data grid: n^(-k/(21*d)), k = 1, ..., 20.
    "krr_m",
    "krr_r",
    "ease_nw",
    "ease_ll",
    "ease_m",
    "ease_r",
    "pi",
]

# EASE splits the labeled sample into K groups.
# Fit on K - 1 groups and predict the remaining group.
K = 5
# EASE uses SS-SIR to combine the input columns into r new variables.
# Fit the regression using these r variables.
r = 2
# Split the training data into cv_folds groups to compare tuning candidates.
# Fit on all but one group and measure prediction error on the remaining group.
# For EASE, this split uses only the K - 1 training groups.
cv_folds = 10

summary_protein = {}
records_protein = {}
for n in labeled_sizes:
    # EASE's SS-SIR splits the response range into H equal-width intervals (slices).
    # n // 5 means n divided by 5, rounded down.
    H = n // 5
    summary_protein[n], records_protein[n] = run_realdata(
        "protein",
        estimators=estimators,
        n=n,
        N=N,
        reps=reps,
        seed=seed,
        K=K,
        r=r,
        H=H,
        cv_folds=cv_folds,
        jobs=jobs,
        run_name=f"protein_n{n}_N{N}",  # Output folder name for this run.
    )

In [ ]:
import pandas as pd

table_protein = ratio_table(pd.concat(summary_protein.values(), ignore_index=True), "protein")
shown_protein = table_protein.map(lambda value: f"{value:.3f}")
shown_protein = shown_protein.mask(
    table_protein.eq(table_protein.max()), "<b>" + shown_protein + "</b>"
)
HTML(shown_protein.to_html(escape=False))